# ArNet2 - Preprocessing and Model Prediction

This notebook demonstrates the workflow for:
1. **Preprocessing R-peaks annotation data** from the **SHDB-AF** dataset for prediction.
2. **Running the trained ArNet2 model** for **AF prediction** using the preprocessed data.

---

### Dataset:
- **SHDB-AF**: The dataset used in this example is the [SHDB-AF dataset](https://physionet.org/content/shdb-af/1.0.1/), which contains ECG signals annotated with peak information and AF-related labels.

### Dataset Details:
- **ECG signals**: The data consists of ECG recordings, where each sample is labeled with corresponding **R-peaks annotations**.
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Each R-peak is annotated with a label indicating whether it is associated with **AF** or not.
- **Overall patient label**: The dataset includes a **global label** for each patient indicating the overall AF status (e.g., **PAF**: Paroxysmal AF, **Per**: Persistent AF, **Non-AF**).

This notebook will help demonstrate how to prepare the data for prediction and how to use the trained **ArNet2 model** to make predictions for **AF detection**.


### 1. Import Libraries

In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

import yaml
import numpy as np
import pandas as pd
import wfdb
import subprocess
from tqdm.notebook import tqdm  # Import tqdm for Jupyter Notebooks
import nb_utils
import run_ArNet2
from pathlib import Path


### 2. Setting Up the Paths
We will first set up paths for data storage and where to save the processed data:

In [2]:
# Setup directories
db_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
input_path = '.././data'  # Where to save the processed data which will be used as input to the model
results_dir = '../results'  # Where to save the predictions
physiozoo_dir = results_dir + '/physiozoo_rhythms'
predictions_dir = results_dir + '/predictions'
statistics_dir = results_dir + '/af_statistics'
model_dir = '../exported_model'  # Where the .pb model is saved

# Create directories if they don't exist
os.makedirs(db_path, exist_ok=True)
os.makedirs(input_path, exist_ok=True)
os.makedirs(physiozoo_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(statistics_dir, exist_ok=True)

window_size=60

### 3. Download the Data
Next, we will download the required ECG signal data and annotation files for SHDB-AF:

In [3]:
# Download annotation files and additional data
nb_utils.download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{db_path}/AdditionalData.csv')

additionaldata = pd.read_csv(f'{db_path}/AdditionalData.csv')
record_names = additionaldata.Data_ID.astype(str).str.zfill(3)

In [4]:
for record_name in tqdm(record_names, desc="Downloading ECG records", leave=False):
    filepath = f'{db_path}/{record_name}'
    url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'
    #  download records and annotations
    if nb_utils.url_exists(f"{url}.atr") and not os.path.exists(f"{filepath}.hea"):
        nb_utils.download_file(f'{url}.atr', f'{filepath}.atr')
        nb_utils.download_file(f'{url}.qrs', f'{filepath}.qrs')
        nb_utils.download_file(f'{url}.hea', f'{filepath}.hea')
        nb_utils.download_file(f'{url}.dat', f'{filepath}.dat')

### 4. Preprocess annotation Data

We will preprocess the R-peaks, time stamps and associated recording ids

In [5]:
all_dfs = []  # list to collect per-record features DataFrames
all_y_df = []  # list to collect per-record true labels Dataframes
for record_name in tqdm(record_names, desc="Processing ECG records", leave=False):
    filepath = f"{db_path}/{record_name}"

    try:
        if not os.path.exists(f"{filepath}.atr"):
            continue
        # Load annotations (e.g., R-peaks)

        else:
            annotation = wfdb.rdann(filepath, 'atr')

            # Extract R-peak sample indices
            r_peaks = annotation.sample

            # Skip files with too few beats
            if len(r_peaks) < 2:
                print(f"Skipping {record_name}: not enough peaks ({len(r_peaks)})")
                continue

            # Calculate RR intervals and convert to seconds
            rr_intervals = np.diff(r_peaks).astype(np.float32)
            rr_time = r_peaks[1:] / annotation.fs
            rr_data = rr_intervals / annotation.fs

            #  derive y labels
            y = nb_utils.calc_y(annotation.aux_note, window_size)

            # Create per-record DataFrames
            ecg_df = pd.DataFrame({
                'rr_data': rr_data,
                'rr_time': rr_time,
                'patient_id': record_name,
            })

            y_df = pd.DataFrame({
                'true_label': y,
                'prec_window': np.arange(y.size, dtype=int),
                'patient_id': record_name,
            })

            # Append to the lists
            all_dfs.append(ecg_df)
            all_y_df.append(y_df)
    except Exception as e:
        print(f"Error processing {record_name}: {e}")
        continue

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_y_df = pd.concat(all_y_df, ignore_index=True)

print(f"Processed overall {combined_df.patient_id.nunique()} patients ..")

Processing ECG records:   0%|          | 0/128 [00:00<?, ?it/s]

Processed overall 98 patients ..


### 5. Save the Preprocessed Data for Prediction
We save the processed R-peak annotation data as a CSV file, which will be used for prediction:

In [6]:
# Save the DataFrame as CSV
input_file = os.path.join(input_path, 'shdb_df_test.csv')
combined_df.to_csv(input_file, index=False)

print(f"ECG data processing complete. File saved as {input_file}")

ECG data processing complete. File saved as .././data/shdb_df_test.csv


### 6. Run Prediction with the Trained Model (Full Architecture (Parts 1 & 2))
Finally, we use the trained ArNet2 model to make predictions based on the preprocessed data:

In [7]:
subprocess.run(['python', '../run_ArNet2.py', '--saved_model', model_dir, '--inference_mode', 'full', '--input_file', input_file, '--output_name', 'shdb_pred_df_full', '--save_output_path', predictions_dir])

Predicting...
File '.././data/shdb_df_test.csv' passed validation.
Results saved to ../results/predictions/shdb_pred_df_full.csv


CompletedProcess(args=['python', '../run_ArNet2.py', '--mode', 'predict', '--inference_mode', 'full', '--input_file', '.././data/shdb_df_test.csv', '--output_name', 'shdb_pred_df_full', '--save_output_path', '../results/predictions', '--config', '/home/shanybiton/repos/Shany_Repo/plug-and-play/config/config_w_abs_path.yml'], returncode=0)

### 8. Run Model Evaluation


This section evaluates the performance of the **ArNet2** model on the prediction results using various performance metrics such as **accuracy**, **F1-score**, **sensitivity**, **specificity**, **AUROC**, and **AUPRC**.
We will compare the model's predicted labels and the true labels to calculate these metrics.

In addition to these metrics, the **mean absolute AF burden (AFB)** error is also evaluated. This metric quantifies the average discrepancy between the predicted and actual AFB across patients, providing insight into the model’s ability to estimate the proportion of time a patient experiences AF. It is calculated by computing the AFB for each patient from predicted and true labels and then averaging the absolute error across all patients.

#### 8.1 Overall model performance

In [8]:
import utils.metrics as metrics
pred_df = pd.read_csv(f"{predictions_dir}/shdb_pred_df_full.csv")
pred_df.patient_id = pred_df.patient_id.astype(str).str.zfill(3)

# Merge true label per window to prediction df
combined_pred_df = pred_df.merge(combined_y_df[['patient_id', 'prec_window', 'true_label']], on=['patient_id', 'prec_window'], how='left')


In [9]:
accuracy, fbeta, sensitivity, specificity, PPV, NPV, AUROC, AUCPR = metrics.model_metrics(combined_pred_df.proba, combined_pred_df.true_label, pred_df.pred, print_metrics=True)

Accuracy: 0.9567569761777737
F1-Score: 0.9115368645827154
Sensitivity: 0.9390413804903087
Specificity: 0.9622674502588743
PPV: 0.8855977133373288
NPV: 0.9806759541925586
AUROC: 0.9827540980781245
AUCPR: 0.9250808430390984
[[126568   4963]
 [  2494  38419]]


In [10]:
# Load data for computing mean AFB error
data = pd.read_csv(input_file)
X, start_win, end_win = run_ArNet2.process_data_for_all_ids(data)

m_e_afb = metrics.mean_abs_afb_error(X, pred_df.pred, combined_pred_df.true_label)
print(f"Mean AFb error: {m_e_afb}")

Mean AFb error: 3.377069640644512


#### 8.2 AF event analysis per patient

In [11]:
# Process each patient
rhythm_stats = []
for i, pat in tqdm(enumerate(pred_df.patient_id.unique()), desc="Processing AF statsictics per recording", leave=False):
    name = f'{db_path}/{pat}'
    print(f'Processing recording: {pat}')
    
    # Load record
    record = wfdb.rdrecord(name)
    
    # Save patient predictions
    temp_pred_file = f'{predictions_dir}/pred_{pat}.csv'
    pred_df.loc[pred_df.patient_id.isin([pat])].to_csv(temp_pred_file, index=False)
    
    # Generate PhysioZoo rhythm file
    output_path = f'{physiozoo_dir}/physiozoo_rhythms_{pat}'
    subprocess.run([
        'python', '../../utils/export_arnet2_output_to_physiozoo.py',
        '--input_file', str(temp_pred_file),
        '--output_name', str(output_path)
    ], stdout=subprocess.DEVNULL)
    
    # Read and analyze rhythm events
    pat_df = pd.read_csv(
        f'{str(output_path)}.txt', 
        sep=r"\s+|;|,", 
        skiprows=6, 
        engine='python'
    )
    
    # Compute AF event statistics
    rec_length = len(record.p_signal) / record.fs
    n_events, max_event, min_event, afb = metrics.compute_af_event_statistics(
        rec_length, pat_df
    )
    
    # Store results
    rhythm_stats.append({
        'patient_id': pat,
        'n_events': n_events,
        'max_event_sec': max_event,
        'min_event_sec': min_event,
        'af_burden_pct': 100 * afb
    })

    os.remove(temp_pred_file)

Processing AF statsictics per recording: 0it [00:00, ?it/s]

Processing recording: 001


Number of AF events: 1,
Longest event: 48592.56 sec,
Shortest event: 48592.56 sec,
AF-burden: 56.44%
Processing recording: 010


Number of AF events: 38,
Longest event: 11839.05 sec,
Shortest event: 50.6 sec,
AF-burden: 64.9%
Processing recording: 102


Number of AF events: 1,
Longest event: 48.19 sec,
Shortest event: 48.19 sec,
AF-burden: 0.06%
Processing recording: 103


...

Number of AF events: 3,
Longest event: 45.22 sec,
Shortest event: 36.04 sec,
AF-burden: 0.14%
Processing recording: 008


Number of AF events: 5,
Longest event: 42932.54 sec,
Shortest event: 41.89 sec,
AF-burden: 68.28%
Processing recording: 084


Number of AF events: 1,
Longest event: 46.6 sec,
Shortest event: 46.6 sec,
AF-burden: 0.05%
Processing recording: 086


Number of AF events: 1,
Longest event: 41.28 sec,
Shortest event: 41.28 sec,
AF-burden: 0.05%
Processing recording: 009


Number of AF events: 2,
Longest event: 10285.9 sec,
Shortest event: 49.38 sec,
AF-burden: 11.96%


In [12]:
# Create summary DataFrame
rhythm_df = pd.DataFrame(rhythm_stats)
print("\nAF Event Statistics Summary:")
print(rhythm_df)

# Save summary
summary_path = f"{statistics_dir}/af_event_summary.csv"
rhythm_df.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")


AF Event Statistics Summary:
   patient_id  n_events  max_event_sec  min_event_sec  af_burden_pct
0         001         1      48592.560      48592.560      56.437352
1         010        38      11839.055         50.595      64.900440
2         102         1         48.185         48.185       0.056876
3         103         1         39.095         39.095       0.045249
4         105         3       4311.465         46.450       5.374039
..        ...       ...            ...            ...            ...
93        077         3         45.220         36.035       0.139132
94        008         5      42932.540         41.890      68.281389
95        084         1         46.600         46.600       0.053935
96        086         1         41.280         41.280       0.047778
97        009         2      10285.905         49.375      11.962130

[98 rows x 5 columns]

Summary saved to: ../results/af_statistics/af_event_summary.csv
